# Phase 1 Task B — Ablation 3.2: Label Smoothing — MobileNetV3-Large

| Item | Value |
|------|-------|
| Experiment | Label Smoothing ablation (0.0 and 0.2) |
| Model | `mobilenetv3_large` |
| Seed | 42 |
| Batch Size | 32 |
| Output | `data/experiments/taskB_label_smoothing_0.0/mobilenetv3_large/` and `data/experiments/taskB_label_smoothing_0.2/mobilenetv3_large/` |
| WandB | `woodbench-taskB` |

In [1]:
# ── Cell 1: Root Detection ────────────────────────────────────
from pathlib import Path
import sys, os

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    NOTEBOOK = Path("__file__").resolve()
    ROOT = NOTEBOOK
    for _ in range(10):
        ROOT = ROOT.parent
        if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break

sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

[ROOT] /home/sakana/Code/ManifoldsTeam/woodbench


In [2]:
# ── Cell 2: Environment Setup ─────────────────────────────────
from dotenv import load_dotenv
dotenv_path = ROOT / ".env"
load_dotenv(dotenv_path)
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops", "python-dotenv"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
os.environ.setdefault("HF_TOKEN", os.getenv("HF_TOKEN", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")
print(f"[HF] token: {'set' if os.getenv('HF_TOKEN') else 'MISSING'}")

[.env] Loaded from /home/sakana/Code/ManifoldsTeam/woodbench/.env
[wandb] API key: set
[HF] token: set


In [3]:
# ── Cell 3: Data Download ─────────────────────────────────────
HF_REPO = "ny0g507h3p/wood-task-B"
DATA_ROOT = ROOT / "data" / "task-B-group-split"

if not (DATA_ROOT / "train").exists():
    print(f"[HF] Downloading dataset to {DATA_ROOT}...")
    from huggingface_hub import login, snapshot_download
    token = os.getenv("HF_TOKEN", "")
    if token:
        login(token=token)
        print("[HF] Logged in")
    else:
        print("[HF] No token — trying public access")
    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        local_dir=str(DATA_ROOT),
        ignore_patterns=["*.md", "README*"],
    )
    print("[HF] Download complete")
else:
    print(f"[Data] Found at {DATA_ROOT}")

# Verify
train_dir = DATA_ROOT / "train"
if not train_dir.exists():
    raise FileNotFoundError(f"[ERROR] train/ not found in {DATA_ROOT}. Dataset download failed.")
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"[Data] {len(classes)} classes: {classes[:3]}...")
print(f"[Data] Train samples: {sum(1 for _ in train_dir.rglob('*.jpg')) + sum(1 for _ in train_dir.rglob('*.png'))}")

[Data] Found at /home/sakana/Code/ManifoldsTeam/woodbench/data/task-B-group-split
[Data] 20 classes: ['Araucaria araucana', 'Callitris columellaris', 'Calocedrus rupestris']...
[Data] Train samples: 3876


In [4]:
# ── Cell 4: DataLoaders ───────────────────────────────────────
from src.data import get_dataloaders
tl, vl, testl = get_dataloaders(
    str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB",
)
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

[Data] Train=3876, Val=903, Test=793
[Data] Classes=20


In [5]:
# ── Cell 5: Imports & Device ──────────────────────────────────
import wandb, torch, numpy as np, json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function, WandbTracker
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

[Device] cuda


In [6]:
# ── Cell 6: train_one (label smoothing ablation) ─────────────
def train_one(backbone, seed, data_root, label_smoothing, batch_size=32, epochs=50):
    """Train one model with a specific label_smoothing value."""
    set_seed(seed)
    exp_name = f"taskB_label_smoothing_{label_smoothing}"
    output_dir = ROOT / "data" / "experiments" / "Task-B" / exp_name / backbone
    output_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, test_loader = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB")
    num_classes = len(train_loader.dataset.classes)
    print(f"[{backbone} s{seed} LS={label_smoothing}] train={len(train_loader.dataset)}, classes={num_classes}")

    tracker = WandbTracker.start(
        project="woodbench-taskB-clean",
        name=f"{backbone}_ls{label_smoothing}_s{seed}",
        config={
            "backbone": backbone, "seed": seed, "epochs": epochs,
            "batch_size": batch_size, "num_classes": num_classes,
            "lr_head": 1e-4, "lr_backbone": 1e-5,
            "label_smoothing": label_smoothing,
            "scheduler": "cosine", "amp": True,
            "experiment": exp_name,
            "ablation": f"label_smoothing={label_smoothing}",
        },
        tags=["ablation", f"label_smoothing_{label_smoothing}", backbone, f"seed{seed}"],
        dir_path=str(output_dir),
    )

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=label_smoothing)

    engine = TrainingEngine(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True)
    result = engine.fit(
        epochs=epochs, output_dir=output_dir,
        name=f"{backbone}_seed{seed}", save_best_only=True,
        early_stopping_patience=10, tracker=tracker)

    # ── Test evaluation ──
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_metrics = {
        "test_accuracy": accuracy_score(all_labels, all_preds),
        "test_f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "test_precision_macro": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "test_recall_macro": recall_score(all_labels, all_preds, average="macro", zero_division=0),
    }

    if tracker:
        tracker.log(
            {
                # Validation / training summary
                "best_epoch": result["best_epoch"],
                "best_val_acc": result["best_val_acc"],
                "epochs_trained": result["epochs_trained"],
                "stopped_early": result["stopped_early"],

                # Test metrics
                **test_metrics,
            },
            step=result["epochs_trained"],
        )

        tracker.finish()

    # Save metrics
    with open(output_dir / "test_metrics.json", "w") as f:
        json.dump(test_metrics, f, indent=2)

    print(f"[{backbone} s{seed} LS={label_smoothing}] val_acc={result['best_val_acc']:.2f}%")
    print(f"[{backbone} s{seed} LS={label_smoothing}] test_acc={test_metrics['test_accuracy']*100:.2f}%")
    return {**result, "test_metrics": test_metrics}

In [ ]:
# ── Cell 7: Run Training (LS=0.0 and LS=0.2) ─────────────────
BACKBONE = "mobilenetv3_large"
SEED = 42
LS_VALUES = [0.0, 0.2]
results = {}

for ls in LS_VALUES:
    print(f"\n{'='*60}")
    print(f"Ablation 3.2: Label Smoothing={ls} — {BACKBONE} | seed={SEED}")
    print(f"{'='*60}")
    results[ls] = train_one(BACKBONE, SEED, str(DATA_ROOT), label_smoothing=ls, batch_size=32, epochs=50)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.



Ablation 3.2: Label Smoothing=0.0 — mobilenetv3_large | seed=42
[mobilenetv3_large s42 LS=0.0] train=3876, classes=20


wandb: Currently logged in as: ny0g507h3p (ny0g507h3p_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/50 | Train Loss: 3.0077, Acc: 5.91% | Val Loss: 3.0168, Acc: 5.54% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.54%)


Epoch 2/50 | Train Loss: 2.8092, Acc: 25.44% | Val Loss: 2.6593, Acc: 34.66% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 34.66%)


Epoch 3/50 | Train Loss: 2.1741, Acc: 60.74% | Val Loss: 1.8562, Acc: 62.13% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 62.13%)


... (77 outputs left out)

Epoch 34/50 | Train Loss: 1.1209, Acc: 99.35% | Val Loss: 1.1871, Acc: 95.79% | LR: 2.81e-05[EarlyStop 9/10]


Epoch 35/50 | Train Loss: 1.1207, Acc: 99.33% | Val Loss: 1.1770, Acc: 96.12% | LR: 2.50e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 35.


best_epoch,▁
best_val_acc,▁
epochs_trained,▁
lr,▁▃▅▆███████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁
test_accuracy,▁
test_f1_macro,▁
test_precision_macro,▁
test_recall_macro,▁
train/acc,▁▂▅▇▇▇█████████████████████████████
train/loss,█▇▆▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...


[mobilenetv3_large s42 LS=0.2] val_acc=97.01%
[mobilenetv3_large s42 LS=0.2] test_acc=95.33%


In [ ]:
# ── Cell 8: Summary ───────────────────────────────────────────
print("\n" + "="*60)
print(f"Ablation 3.2: Label Smoothing — {BACKBONE} — Summary")
print("="*60)
for ls, result in results.items():
    tm = result['test_metrics']
    print(f"\n  Label Smoothing = {ls}")
    print(f"  Seed:            {SEED}")
    print(f"  Val Acc (best):  {result['best_val_acc']:.2f}%")
    print(f"  Best Epoch:      {result['best_epoch']}")
    print(f"  Epochs Trained:  {result['epochs_trained']}")
    print(f"  Early Stopped:   {result['stopped_early']}")
    print(f"  ---")
    print(f"  Test Accuracy:   {tm['test_accuracy']*100:.2f}%")
    print(f"  Test F1 (macro): {tm['test_f1_macro']*100:.2f}%")
    print(f"  Test Precision:  {tm['test_precision_macro']*100:.2f}%")
    print(f"  Test Recall:     {tm['test_recall_macro']*100:.2f}%")
    print(f"  Output: data/experiments/taskB_label_smoothing_{ls}/{BACKBONE}/")


Ablation 3.2: Label Smoothing — mobilenetv3_large — Summary

  Label Smoothing = 0.0
  Seed:            42
  Val Acc (best):  97.34%
  Best Epoch:      35
  Epochs Trained:  45
  Early Stopped:   True
  ---
  Test Accuracy:   94.45%
  Test F1 (macro): 90.43%
  Test Precision:  89.71%
  Test Recall:     92.00%
  Output: data/experiments/taskB_label_smoothing_0.0/mobilenetv3_large/

  Label Smoothing = 0.2
  Seed:            42
  Val Acc (best):  97.01%
  Best Epoch:      25
  Epochs Trained:  35
  Early Stopped:   True
  ---
  Test Accuracy:   95.33%
  Test F1 (macro): 90.91%
  Test Precision:  89.94%
  Test Recall:     92.38%
  Output: data/experiments/taskB_label_smoothing_0.2/mobilenetv3_large/
